# B.1 Grundspecifikation och årseffekt

Syftet med denna notebook är att bekräfta variabelvalet från den deskriptiva analysen (A4) genom att bygga en sekvens av Poisson-GLM-modeller och jämföra dem med AIC och valideringsdeviance.

Fyra modeller byggs i sekvens — från en enkel null-baseline till en modell med årseffekt. Modell M2 förväntas bli slutmodellen.

**Upplägg:**
- Träning: 2021–2023
- Validering: 2024
- Test 2025 reserverad för Fas D

**Kontext från A4:** Omsättning valdes som storleksvariabel (starkast monotont samband med skadefrekvens, 5.6× spridning mellan deciler). Försäkringsbelopp och Självrisk exkluderades på grund av korrelation med omsättning respektive svagt samband.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

data_dir = Path("../../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")

print(f"Träning: {df.shape[0]:,} rader, {df.shape[1]} kolumner")

Träning: 1,033,386 rader, 8 kolumner


In [2]:
# Datakontroll och förberedelse
assert (df["Duration"] > 0).all(), "Duration innehåller nollor — hantera innan log"

df["Ar"] = df["Ar"].astype(int)
df["log_duration"] = np.log(df["Duration"])
df["log_omsattning"] = np.log(df["Omsattning"])

# Temporal split: träna 2021-2023, validera 2024
df_train = df[df["Ar"].isin([2021, 2022, 2023])].copy()
df_val = df[df["Ar"] == 2024].copy()

print(f"Träning (2021-2023): {df_train.shape[0]:,} rader")
print(f"Validering (2024):   {df_val.shape[0]:,} rader")

Träning (2021-2023): 755,691 rader
Validering (2024):   277,695 rader


## Modellspecifikation

Alla modeller använder Poisson-familj med log-länk och `log(Duration)` som offset. Referenskategorier: **Byggföretag** (Verksamhet) och **Landsbyggd** (GeografisktOmrade).

| Modell | Specifikation | Syfte |
|--------|--------------|-------|
| M0 | Intercept only + offset | Null-baseline som referens |
| M1 | C(Verksamhet) + C(GeografisktOmrade) + offset | Kategoriska huvudeffekter |
| M2 | M1 + log(Omsättning) + offset | **Primär slutmodell** |
| M3 | M2 + C(Ar) + offset | Årseffekt, enbart in-sample |

In [3]:
# M0: Intercept only (null-baseline)
m0 = smf.glm(
    "AntalSkador ~ 1",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M0  AIC: {m0.aic:,.2f}   Deviance: {m0.deviance:,.2f}")

M0  AIC: 141,086.49   Deviance: 112,768.79


In [4]:
# M1: Kategoriska huvudeffekter
m1 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd'))",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M1  AIC: {m1.aic:,.2f}   Deviance: {m1.deviance:,.2f}")

M1  AIC: 139,723.38   Deviance: 111,387.69


In [5]:
# M2: Primär slutmodell — lägger till log(Omsättning)
m2 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M2  AIC: {m2.aic:,.2f}   Deviance: {m2.deviance:,.2f}")

M2  AIC: 136,971.03   Deviance: 108,633.33


In [6]:
# M3: Årseffekt (enbart in-sample — kan inte prediktera osedda år)
m3 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning "
    "+ C(Ar)",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M3  AIC: {m3.aic:,.2f}   Deviance: {m3.deviance:,.2f}")

M3  AIC: 136,968.88   Deviance: 108,627.19


## 1. AIC-jämförelse M0–M3

AIC (Akaike Information Criterion) balanserar modellanpassning mot komplexitet — lägre AIC är bättre. Alla fyra modeller jämförs in-sample.

In [7]:
aic_table = pd.DataFrame({
    "Modell": ["M0", "M1", "M2", "M3"],
    "Beskrivning": [
        "Intercept only",
        "Verksamhet + Geografi",
        "M1 + log(Omsättning)",
        "M2 + År",
    ],
    "Antal parametrar": [
        m0.df_model + 1,
        m1.df_model + 1,
        m2.df_model + 1,
        m3.df_model + 1,
    ],
    "AIC": [m0.aic, m1.aic, m2.aic, m3.aic],
    "Deviance": [m0.deviance, m1.deviance, m2.deviance, m3.deviance],
})

display(aic_table.style.format({"AIC": "{:,.0f}", "Deviance": "{:,.0f}"}))

,Modell,Beskrivning,Antal parametrar,AIC,Deviance
0,M0,Intercept only,1,"141,086","112,769"
1,M1,Verksamhet + Geografi,10,"139,723","111,388"
2,M2,M1 + log(Omsättning),11,"136,971","108,633"
3,M3,M2 + År,13,"136,969","108,627"


### Tolkning

Förväntad rangordning:

- **M0 → M1:** Stort AIC-fall. Verksamhet och geografi fångar tydliga skillnader i skadefrekvens (bekräftar A3:s deskriptiva resultat).
- **M1 → M2:** Ytterligare AIC-förbättring. Omsättning (log-transformerad) tillför prediktiv information utöver kategoriska variabler.
- **M2 → M3:** Marginell förbättring. Årseffekt fångar mindre tidsvariation men tillför få parametrar.

AIC avgör in-sample. Nästa steg är att verifiera på osedd data (2024).

## 2. Valideringsdeviance M0–M2

AIC mäter in-sample-anpassning. Valideringsdeviance på 2024-data bekräftar att förbättringarna generaliserar till osedd data.

M3 utesluts här — `C(Ar)` är kategorisk och kan inte prediktera 2024, som är en osedd nivå.

In [8]:
import warnings


def poisson_deviance(y_true, y_pred):
    """Beräkna Poisson-deviance mellan observerade och predikterade värden."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 1e-12, None)

    # Beräkna bidrag separat för nollor och icke-nollor
    mask = y_true > 0
    dev_nonzero = np.sum(y_true[mask] * np.log(y_true[mask] / y_pred[mask]))
    dev = 2 * (dev_nonzero - np.sum(y_true - y_pred))
    return dev


# Prediktera på valideringsdata (2024)
y_val = df_val["AntalSkador"].values
val_offset = df_val["log_duration"]

val_pred_m0 = m0.predict(df_val, offset=val_offset)
val_pred_m1 = m1.predict(df_val, offset=val_offset)
val_pred_m2 = m2.predict(df_val, offset=val_offset)

val_dev_table = pd.DataFrame({
    "Modell": ["M0", "M1", "M2"],
    "Beskrivning": [
        "Intercept only",
        "Verksamhet + Geografi",
        "M1 + log(Omsättning)",
    ],
    "Valideringsdeviance (2024)": [
        poisson_deviance(y_val, val_pred_m0.values),
        poisson_deviance(y_val, val_pred_m1.values),
        poisson_deviance(y_val, val_pred_m2.values),
    ],
    "Totalt predikterat": [
        val_pred_m0.sum(),
        val_pred_m1.sum(),
        val_pred_m2.sum(),
    ],
    "Totalt observerat": [
        y_val.sum(),
        y_val.sum(),
        y_val.sum(),
    ],
})

display(val_dev_table.style.format({
    "Valideringsdeviance (2024)": "{:,.0f}",
    "Totalt predikterat": "{:,.0f}",
    "Totalt observerat": "{:,.0f}",
}))

,Modell,Beskrivning,Valideringsdeviance (2024),Totalt predikterat,Totalt observerat
0,M0,Intercept only,"42,643","5,250","5,446"
1,M1,Verksamhet + Geografi,"42,105","5,247","5,446"
2,M2,M1 + log(Omsättning),"41,002","5,250","5,446"


### Tolkning

M2 har lägst valideringsdeviance — förbättringen från att lägga till `log(Omsättning)` generaliserar till osedd data. Kolumnen "Totalt predikterat" visar hur väl varje modell träffar det faktiska skadeantalet i 2024-portföljen.

Slutsats: M2 bekräftas som bästa modell av både AIC (in-sample) och valideringsdeviance (out-of-sample).

## 3. M2-koefficienter och rate ratios

Rate ratio = exp(β). Visar den multiplikativa effekten på skadefrekvens jämfört med referenskategorin. Ett rate ratio på 1.5 innebär 50 % högre förväntad frekvens.

In [9]:
conf = m2.conf_int()

coef_table = pd.DataFrame({
    "Koefficient (β)": m2.params,
    "Std.fel": m2.bse,
    "Rate ratio": np.exp(m2.params),
    "KI nedre (2.5%)": np.exp(conf.iloc[:, 0]),
    "KI övre (97.5%)": np.exp(conf.iloc[:, 1]),
})

display(coef_table.round(4))

,Koefficient (β),Std.fel,Rate ratio,KI nedre (2.5%),KI övre (97.5%)
Intercept,-11.0790,0.1401,0.0000,0.0000,0.0000
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Elektriker]",-0.3713,0.0339,0.6898,0.6455,0.7372
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Grävning & Schaktning]",-0.1543,0.0339,0.8570,0.8018,0.9159
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Målare]",-0.4448,0.0349,0.6409,0.5986,0.6863
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Takarbeten]",0.1029,0.0373,1.1084,1.0301,1.1926
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.VVS]",0.3557,0.0254,1.4272,1.3579,1.5000
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Övriga specialistföretag]",-0.0193,0.0240,0.9809,0.9358,1.0281
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Mellanstorstad]",0.1858,0.0327,1.2041,1.1294,1.2838
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Småstad]",-0.2788,0.0373,0.7567,0.7034,0.8141
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Storstad]",0.3796,0.0313,1.4618,1.3748,1.5543


### Kort tolkning

Koefficienterna och rate ratios tolkas i detalj i B2 (modellkontroll och tolkning). Notera redan här:

- **Verksamhet:** Skillnader mellan verksamhetstyper relativt Byggföretag — förväntad rangordning med VVS högst och Målare lägst (konsistent med A3).
- **Geografi:** Storstad förväntas ha högst rate ratio, konsistent med den deskriptiva analysen.
- **log(Omsättning):** Positivt samband — större företag har högre skadefrekvens (utöver vad Duration redan fångar).

## 4. Årseffekt (M3, enbart in-sample)

M3 inkluderar `C(Ar)` som kategorisk variabel. Eftersom Ar har tre nivåer i träningsdata (2021, 2022, 2023) kan modellen inte prediktera osedda år som 2024 eller 2025 — en osedd kategorinivå ger ingen prediktion.

Syftet med M3 är enbart att undersöka om skadefrekvensen varierat mellan åren, inte att förbättra prediktion.

In [10]:
# Extrahera enbart årskoefficienter från M3
year_mask = m3.params.index.str.contains("C(Ar)", regex=False)
year_params = m3.params[year_mask]
year_conf = m3.conf_int().loc[year_mask]

year_table = pd.DataFrame({
    "Koefficient (β)": year_params.values,
    "Rate ratio": np.exp(year_params.values),
    "KI nedre (2.5%)": np.exp(year_conf.iloc[:, 0].values),
    "KI övre (97.5%)": np.exp(year_conf.iloc[:, 1].values),
}, index=year_params.index)

# Renare radnamn
year_table.index = year_table.index.str.extract(r"\[T\.(\d+)\]")[0].values
year_table.index.name = "År (ref: 2021)"

display(year_table.round(4))

,Koefficient (β),Rate ratio,KI nedre (2.5%),KI övre (97.5%)
År (ref: 2021),,,,
2022,-0.0396,0.9612,0.9226,1.0013
2023,0.0077,1.0078,0.9683,1.0488


### Tolkning

Årseffekterna visar om skadefrekvensen förändrats relativt 2021 efter kontroll för verksamhet, geografi och omsättning. En svag uppåtgående trend är konsistent med den deskriptiva analysen (A3).

Eftersom årseffekten är kategorisk och inte kan extrapoleras till framtida år bör M3 **inte** användas som prediktiv modell. Tidsvariationen är intressant att beskriva i rapporten men den förändrar inte modellvalet.

## 5. Slutsats: M2 som slutmodell

**M2 väljs som primär Poisson-GLM** baserat på:

1. **Lägst AIC** bland M0–M2 — varje steg (kategoriska variabler, sedan omsättning) förbättrar anpassningen.
2. **Lägst valideringsdeviance på 2024** — förbättringarna generaliserar till osedd data.
3. **M3 utesluts** som prediktiv modell — årseffekten är informativ men kan inte prediktera framtida år.

M2-specifikation:

```
AntalSkador ~ C(Verksamhet) + C(GeografisktOmrade) + log(Omsättning) + offset(log(Duration))
```

**Nästa steg:** I B2 omskolas M2 på hela träningsdatan (2021–2024) för att ge bästa möjliga koefficientskattningar. Där görs överdispersionskontroll, detaljerad tolkning av rate ratios och affärsmässig översättning.